# System preferences
Here we use the user answers to the comparative questions from the user study. We will evaluate the results for each aspect and analyze the results based on binominal test and carry-over effects

In [1]:
import pandas as pd
import scipy.stats as st

In [2]:
# import sheet 
df = pd.read_excel('../UserStudyCoding.xlsx', sheet_name='comparative_sys_preference')

In [3]:
preference_categories = [
    "overall_preference",
    "more_efficient",
    "less_frustrating",
    "more_level_adapted",
    "more_task_adapted",
    "better_mistake_handling",
    "more_productive",
    "more_confident",
    "experiment_guess"
]

order_col = "group"

results = []

for col in preference_categories: 
    # ensure that all cells have the same spelling
    df[col] = df[col].astype(str).str.strip().str.lower()
    
    # sum up the preferences
    n_exp = (df[col] == "experiment").sum()
    n_ctrl = (df[col] == "control").sum()
    n_neu = (df[col] == "neutral").sum()
    
    n_decided = n_exp + n_ctrl
    n_total = n_decided + n_neu

    # 2. Exact binominal test => Only on the decided votes
    if n_decided > 0:
        binom_res = st.binomtest(k=n_exp, n=n_decided, p=0.5, alternative='two-sided')
        p_binom = binom_res.pvalue
    else:
        p_binom = float('nan')

    # Fisher's Exact Test (Carry-Over Check)
    # we filter out the neutral votes for the carry-over check
    df_decided = df[df[col].isin(["experiment", "control"])]
    
    if len(df_decided) > 0:
        # Cross table: Group vs. Vote
        crosstab = pd.crosstab(df_decided[order_col], df_decided[col])
        
        # Make sure its a 2x2 matrix
        for g in ["control_first", "experiment_first"]:
            if g not in crosstab.index:
                crosstab.loc[g] = 0
        for v in ["experiment", "control"]:
            if v not in crosstab.columns:
                crosstab[v] = 0
                
        # Fisher Test on 2x2 matrix
        odds_ratio, p_fisher = st.fisher_exact(crosstab.loc[["control_first", "experiment_first"], ["experiment", "control"]])
    else:
        p_fisher = float('nan')

    # Format results
    p_binom_clean = "< .001" if p_binom < 0.001 else f"{p_binom:.3f}".lstrip("0") if pd.notna(p_binom) else "N/A"
    p_fisher_clean = "< .001" if p_fisher < 0.001 else f"{p_fisher:.3f}".lstrip("0") if pd.notna(p_fisher) else "N/A"

    results.append({
        "Category": col,
        "Total (N)": n_total,
        "Neutral Votes": f"{n_neu} ({n_neu/n_total*100:.0f}%)",
        "Experiment Votes": f"{n_exp} ({n_exp/n_total*100:.0f}%)",
        "Control Votes": f"{n_ctrl} ({n_ctrl/n_total*100:.0f}%)",
        "Decided (n)": n_decided,
        "Binomial p": p_binom_clean,
        "Carry-Over p": p_fisher_clean
    })

df_results = pd.DataFrame(results)
df_results

,Category,Total (N),Neutral Votes,Experiment Votes,Control Votes,Decided (n),Binomial p,Carry-Over p
0,overall_preference,20,1 (5%),10 (50%),9 (45%),19,1.000,.656
1,more_efficient,20,7 (35%),8 (40%),5 (25%),13,.581,.293
2,less_frustrating,20,5 (25%),8 (40%),7 (35%),15,1.000,1.000
3,more_level_adapted,20,12 (60%),4 (20%),4 (20%),8,1.000,.486
4,more_task_adapted,20,8 (40%),6 (30%),6 (30%),12,1.000,1.000
5,better_mistake_handling,20,8 (40%),7 (35%),5 (25%),12,.774,.242
6,more_productive,20,5 (25%),7 (35%),8 (40%),15,1.000,1.000
7,more_confident,20,6 (30%),5 (25%),9 (45%),14,.424,.580
8,experiment_guess,20,3 (15%),10 (50%),7 (35%),17,.629,.622


In [4]:
df_results.to_csv("./system_preferences.csv", index=False, sep=';', encoding='utf-8')